# Aşama 12: Sezonlu model ve spektrum ölçümü

Şimdiye kadar 10 yılı **tek bir dedektörmüş gibi** analiz ettik. Oysa IceCube 2008–2011 arasında inşaat hâlindeydi: 40, 59, 79 ip, sonra tam dedektör (86 ip). Her konfigürasyonun hassasiyeti, enerji ölçümü ve olay seçimi farklı.

Bu aşamada:
1. **Her sezonu ayrı modelliyoruz:** kendi arka planı, kendi enerji tabloları, kendi etkin alanı. Bir kaynağın sinyali sezonlara, o sezonun hassasiyeti ve süresiyle orantılı dağılıyor.
2. **Spektrum eğimi γ'yı daha geniş aralıkta (1.5–4.5) ve daha ince adımla** ölçüyoruz, belirsizliğiyle birlikte.
3. **İki ince düzeltme** (closure testinde fark ettik): sinyalin enerji dağılımını olayın ölçülen yönünde değil **kaynağın yönünde** değerlendiriyoruz (sinyalin enerjisi gerçek geliş yönüne bağlı), ve −5°'deki seçim sınırının çevresinde enerji tablolarını daha ince dilimlere bölüyoruz. Bu ikisi olmadan ekvator yakınındaki kaynaklarda γ sistematik olarak ~0.3 yüksek çıkıyordu.

Aşama 9'da NGC 1068'in γ'sı aralığın sınırına (4.0) dayanmıştı; bunun tek-dedektör sadeleştirmesinden kaynaklanıp kaynaklanmadığını göreceğiz.

In [ ]:
import sys; sys.path.append('../src')
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import icecube_ps as ps
importlib.reload(ps)
assert float(ps.SURUM) >= float('12.0'), f"src/icecube_ps.py eski sürüm ({getattr(ps, 'SURUM', 'yok')}). Yenisini kopyala, kernel'i yeniden başlat."
try:
    from tqdm.auto import tqdm          # ilerleme çubuğu
except ImportError:
    tqdm = None

pd.set_option('display.float_format', '{:.3g}'.format)
DATA, RESULTS = Path('../data'), Path('../results')
ev = ps.veri_yukle(DATA)
B = ps.ArkaPlan(ev['dec'].values)
print('icecube_ps', ps.SURUM, '| olay:', f'{len(ev):,}')
from scipy import stats
sig0 = lambda p: max(0.0, ps.sigma(p))

In [ ]:
alanlar = ps.etkin_alanlari_yukle(DATA)
sureler = ps.canli_sureler_yukle(DATA)
smearingler = ps.smearingleri_yukle(DATA)
M = ps.SezonluModel(ev, alanlar, sureler, smearingler)
pd.DataFrame({'olay': [M.N[s] for s in M.sezonlar],
              'canlı süre (gün)': [M.T[s] / 86400 for s in M.sezonlar],
              'tepki tablosu': [ps._irf_sezonu(s, alanlar) for s in M.sezonlar]}, index=M.sezonlar)

## 1. Önce doğrulama: γ'yı geri bulabiliyor muyuz?

Bilinen spektrumla (γ = 2 ve γ = 3) sahte kaynaklar ekleyip modelin γ'yı doğru ölçüp ölçmediğine bakıyoruz (closure test). Sahte olaylar da sezonlara, her sezonun kendi tablolarına göre dağıtılıyor.

In [ ]:
L_test = ps.SezonluLikelihood(M, 180.0, 0.0)          # NGC 1068 ile aynı Dec, boş bir nokta
rng = np.random.default_rng(12)
satirlar = []
for g_enj in (2.0, 3.0):
    sonuc = np.array([L_test.sinyal_denemesi(50, g_enj, rng) for _ in range(30)])
    satirlar.append(dict(eklenen_gamma=g_enj, eklenen_ns_ort=50,
                         bulunan_ns_ort=sonuc[:, 0].mean(), bulunan_gamma_ort=sonuc[:, 1].mean(),
                         gamma_sapma=sonuc[:, 1].std()))
pd.DataFrame(satirlar)

## 2. Dört kaynak: sezonlu fit

Her kaynak için en iyi nₛ, γ ve belirsizlikleri. γ'nın belirsizliği **profil likelihood** ile: her γ için en iyi nₛ'i bulup TS'nin en iyi değerden 1 birim düştüğü aralık.

In [ ]:
modeller, fitler, profiller = {}, [], {}
for ad, (rk, dk) in ps.KAYNAKLAR.items():
    L = ps.SezonluLikelihood(M, rk, dk)
    modeller[ad] = L
    (ns, gam, ts), tablo = L.fit(tum_gamma=True)
    profiller[ad] = tablo
    icinde = M.gamma[tablo[:, 1] >= ts - 1]                       # γ için 1σ aralığı
    kon = L.kontur(np.linspace(0, 200, 401))
    g_en = int(np.argmin(np.abs(M.gamma - gam)))
    ns_ic = np.linspace(0, 200, 401)[kon[g_en] <= 1]
    N1 = M.aki_basina_olay(dk, gam)
    fitler.append(dict(kaynak=ad, ns=ns, ns_alt=ns_ic.min(), ns_ust=ns_ic.max(),
                       gamma=gam, gamma_alt=icinde.min(), gamma_ust=icinde.max(), TS=ts,
                       E2F_1TeV=ps.e2_aki(ns / N1)))
tablo12 = pd.DataFrame(fitler).set_index('kaynak')
tablo12

In [ ]:
ad = 'NGC 1068'
L = modeller[ad]
ns_izg = np.linspace(0, 160, 321)
kon = L.kontur(ns_izg)
f = tablo12.loc[ad]
fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(ns_izg, M.gamma, kon, levels=[0, 2.30, 5.99], colors=['#d1495b', '#f2b5bd'], alpha=0.8)
ax.contour(ns_izg, M.gamma, kon, levels=[2.30, 5.99], colors='#333', linewidths=1)
ax.plot(f['ns'], f['gamma'], '*', color='white', mec='#333', ms=18, label='en iyi fit')
ax.set_xlabel('nₛ (10 yılda kaynaktan gelen nötrino sayısı)'); ax.set_ylabel('spektrum eğimi γ')
ax.set_title(f'{ad}: nₛ – γ güven bölgeleri (koyu %68, açık %95)')
ax.legend(); ax.grid(alpha=0.3)
plt.savefig(RESULTS / 'asama12_ngc1068_kontur.png', dpi=150, bbox_inches='tight')
plt.show()

**Konturu okumak:** nₛ ve γ birbirine bağlı. Daha yumuşak spektrum (büyük γ) daha çok düşük enerjili olay demek; model bunu daha fazla nₛ ile telafi edebiliyor. Bu yüzden bölge eğik bir muz şeklinde. IceCube makalelerinde de NGC 1068 için tam bu tür bir grafik var.

## 3. Anlamlılık

⏱️ Kaynak başına 1000 karıştırma.

In [ ]:
for f in fitler:
    ad = f['kaynak']
    dosya = RESULTS / f"ts_arka_sezonlu_{ad.replace(' ', '_').replace('+', 'p')}.npy"
    if dosya.exists():
        ts_a = np.load(dosya)
    else:
        ts_a = modeller[ad].arka_plan_denemeleri(1000, np.random.default_rng(1200), ilerleme=tqdm)
        np.save(dosya, ts_a)
    eta, sd = np.mean(ts_a > 0), np.mean(ts_a[ts_a > 0])
    p_mc = (np.sum(ts_a >= f['TS']) + 1) / (len(ts_a) + 1)
    p_k = eta * stats.chi2.sf(f['TS'], df=sd) if f['TS'] > 0 else 1.0
    f.update(sigma_MC=sig0(p_mc), sigma=sig0(p_k))

tablo12 = pd.DataFrame(fitler).set_index('kaynak')
tablo12.to_csv(RESULTS / 'asama12_sezonlu.csv')
a9 = pd.read_csv(RESULTS / 'asama9_enerji.csv').set_index('kaynak')
a3 = pd.read_csv(RESULTS / 'asama3_likelihood.csv').set_index('kaynak')
karsilastirma = pd.DataFrame({
    'σ yön (A3)': a3['sigma_Wilks'], 'σ yön+enerji (A9)': a9['sigma_kuyruk'], 'σ sezonlu (A12)': tablo12['sigma'],
    'γ (A9)': a9['gamma'], 'γ sezonlu (A12)': tablo12['gamma'],
    'γ aralığı': [f"{r.gamma_alt:.1f}–{r.gamma_ust:.1f}" for r in tablo12.itertuples()]})
karsilastirma

## Yorumlarken

- **Closure testi** γ'yı doğru geri buluyorsa, ölçtüğümüz γ değerlerine güvenebiliriz.
- **NGC 1068'in γ'sı** artık aralığın sınırına dayanmıyorsa, Aşama 9'daki tuhaflık gerçekten tek-dedektör sadeleştirmesinden kaynaklanıyormuş demektir.
- **γ aralıkları geniş:** 10 yılda birkaç düzine nötrinodan spektrum ölçmek zor; belirsizliği dürüstçe vermek, tek bir sayı vermekten daha değerli.
- Bundan sonraki aşamalar (enerjili tarama, sağlamlık, karşılaştırma) bu sezonlu modeli kullanacak.